In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [ ]:

DATASET_PATH = "dataset/pneumonia_dataset/dataset"
IMG_SIZE = 128  
def load_dataset(dataset_path):
    images = []
    labels = []
    categories = ["Normal_CT", "Pneumonia_CT"]
    
    for label, category in enumerate(categories):
        folder_path = os.path.join(dataset_path, category)
        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) 
            if img is not None:
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) 
                images.append(img)
                labels.append(label)

    return np.array(images), np.array(labels)


X_train, y_train = load_dataset(os.path.join(DATASET_PATH, "train"))
X_test, y_test = load_dataset(os.path.join(DATASET_PATH, "test"))

X_train = X_train.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
X_test = X_test.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0


y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

print(f"Train dataset shape: {X_train.shape}, Train labels shape: {y_train.shape}")
print(f"Test dataset shape: {X_test.shape}, Test labels shape: {y_test.shape}")


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# Define model
model = Sequential([
    # First Convolutional Block
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(128, 128, 1)),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # Second Convolutional Block
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # Third Convolutional Block
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # Fourth Convolutional Block (New)
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # Fifth Convolutional Block (New)
    Conv2D(512, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2,2)),

    # Flattening
    Flatten(),

    # Fully Connected Layers
    Dense(512, activation='relu'),
    Dropout(0.5),
    
    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(128, activation='relu'),

    # Output Layer
    Dense(2, activation='softmax')  # 2 classes: Normal & Pneumonia
])

# Compile Model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display Model Summary
model.summary()


In [ ]:
# Train model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=70, batch_size=32)

# Save model
model.save("pneumonia_detector.h5")


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.title("Loss")

plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Predict on the test set
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)  # Convert probabilities to class labels
y_true_classes = np.argmax(y_test, axis=1)  # True class labels

# Calculate accuracy
accuracy = accuracy_score(y_true_classes, y_pred_classes)
report = classification_report(y_true_classes, y_pred_classes, target_names=["Normal", "Pneumonia"])

# Print and save results
results_text = f"Model Evaluation Results:\n\n"
results_text += f"Accuracy: {accuracy:.4f}\n\n"
results_text += f"Classification Report:\n{report}\n"

print(results_text)

# Save results to a text file
with open("results.txt", "w") as file:
    file.write(results_text)

# Plot confusion matrix
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "Pneumonia"], yticklabels=["Normal", "Pneumonia"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# Save confusion matrix
plt.savefig("confusion_matrix.png")


In [ ]:
from sklearn.metrics import roc_curve, auc

def plot_roc_curve(y_true, y_probs):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color="blue", label=f"ROC curve (area = {roc_auc:.2f})")
    plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Random classifier line
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend(loc="lower right")
    plt.show()
plot_roc_curve(y_true_classes, y_pred_classes)